# Fusion Branch — Late Fusion, Cross-Attention, and Agreement Explanations

[![GitHub](https://img.shields.io/badge/GitHub-manasdutta04%2Fmultimodal--fake--news--detector-181717?logo=github&logoColor=white)](https://github.com/manasdutta04/multimodal-fake-news-detector)

This notebook is the **multimodal** path of *Multimodal Fake News Detector with Explainability*. It freezes the Module 01 DistilBERT and Module 02 ResNet50 encoders, trains fusion heads on paired text+image embeddings, and produces a text–image agreement score with a short natural-language explanation.

> **Note.** Research prototype — not a deployed fact-checker. Agreement is a signal about modality consistency, not external claim verification.

**Scope**
- Inputs: official Fakeddit splits + Drive image cache from Module 02
- Encoders: frozen DistilBERT (768-d) + frozen ResNet50 (2048-d)
- Fusion: late concat MLP (required); cross-attention fusion (recommended)
- Explainability: cosine agreement + rule-based modality attribution line
- Success check: multimodal test metrics should beat both unimodal baselines

**Prerequisites on Drive (`DATA_DIR`)**
- `checkpoints/module01_distilbert/` (HF weights + tokenizer)
- `checkpoints/module02_resnet50/model.pt`
- `image_cache/` (from Module 02 downloads)
- three `multimodal_*.tsv` files


## Environment

GPU recommended for embedding extraction. Fusion-head training is light and can run on CPU once embeddings are cached.


## Dependencies


In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib seaborn pillow tqdm
%pip install -q "transformers>=4.40"

# PyTorch / torchvision ship with Colab/Kaggle GPU images.


> **Note.** After installing on a fresh runtime, restart once, then continue from imports.


In [ ]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50

from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


## Paths

Use the same Drive folder as Modules 01–02 (typically `/content/drive/MyDrive/dataset`).


In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/dataset"
else:
    DATA_DIR = "/content/dataset"

TRAIN_PATH = os.path.join(DATA_DIR, "multimodal_train.tsv")
VAL_PATH = os.path.join(DATA_DIR, "multimodal_validate.tsv")
TEST_PATH = os.path.join(DATA_DIR, "multimodal_test_public.tsv")
IMAGE_CACHE = os.path.join(DATA_DIR, "image_cache")

TEXT_CKPT = os.path.join(DATA_DIR, "checkpoints", "module01_distilbert")
IMAGE_CKPT = os.path.join(DATA_DIR, "checkpoints", "module02_resnet50", "model.pt")
M1_METRICS = os.path.join(DATA_DIR, "checkpoints", "module01_metrics.csv")
M2_METRICS = os.path.join(DATA_DIR, "checkpoints", "module02_metrics.csv")
EMB_CACHE = os.path.join(DATA_DIR, "checkpoints", "module03_embeddings")
os.makedirs(EMB_CACHE, exist_ok=True)

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, TEXT_CKPT, IMAGE_CKPT, IMAGE_CACHE]:
    assert os.path.exists(p), f"Missing required path: {p}"
print("All Module 01/02 artifacts found.")


## Pair text rows with cached images

Only samples that have both a title and a successfully cached image enter fusion. Subsample for hosted-GPU / first runs; set to `None` for fuller report numbers.


In [ ]:
LABEL_COL = "2_way_label"
TEXT_COL = "clean_title"
ID_COL = "id"
URL_COL = "image_url"
LABEL_NAMES = ["fake", "real"]

SUBSAMPLE_TRAIN = 6_000
SUBSAMPLE_VAL = 1_500
SUBSAMPLE_TEST = 1_500


def cache_path_for(sample_id: str) -> str:
    safe = str(sample_id).replace("/", "_")
    return os.path.join(IMAGE_CACHE, f"{safe}.jpg")


def load_split(path: str) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", low_memory=False)


def with_cached_images(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out[ID_COL] = out[ID_COL].astype(str)
    out["image_path"] = out[ID_COL].map(cache_path_for)
    out = out[out["image_path"].map(os.path.exists)].copy()
    out = out[out[TEXT_COL].notna() & (out[TEXT_COL].astype(str).str.strip() != "")]
    return out.reset_index(drop=True)


def stratified_subsample(df: pd.DataFrame, n):
    if n is None or n >= len(df):
        return df.reset_index(drop=True)
    parts = []
    per_class = max(1, n // 2)
    for _, g in df.groupby(LABEL_COL):
        parts.append(g.sample(n=min(len(g), per_class), random_state=SEED))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=SEED)
    return out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


train_df = stratified_subsample(with_cached_images(load_split(TRAIN_PATH)), SUBSAMPLE_TRAIN)
val_df = stratified_subsample(with_cached_images(load_split(VAL_PATH)), SUBSAMPLE_VAL)
test_df = stratified_subsample(with_cached_images(load_split(TEST_PATH)), SUBSAMPLE_TEST)

print(f"Paired samples → train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
assert len(train_df) and len(val_df) and len(test_df), "Empty paired split — check image_cache."
print(train_df[LABEL_COL].value_counts().sort_index())


## Load frozen encoders

Text: HuggingFace DistilBERT classification checkpoint from Module 01.  
Image: ResNet50 `state_dict` from Module 02. Both stay frozen while fusion heads train.


In [ ]:
# --- Text encoder ---
tokenizer = AutoTokenizer.from_pretrained(TEXT_CKPT)
text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_CKPT)
text_model.to(DEVICE).eval()
for p in text_model.parameters():
    p.requires_grad = False

TEXT_DIM = text_model.config.dim if hasattr(text_model.config, "dim") else 768
MAX_LEN = 64

# --- Image encoder ---
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224


def build_resnet50(num_classes: int = 2) -> nn.Module:
    try:
        from torchvision.models import ResNet50_Weights
        m = resnet50(weights=None)
    except Exception:
        m = resnet50(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


try:
    img_ckpt = torch.load(IMAGE_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    img_ckpt = torch.load(IMAGE_CKPT, map_location="cpu")
image_clf = build_resnet50(num_classes=2)
image_clf.load_state_dict(img_ckpt["model_state_dict"])
image_clf.to(DEVICE).eval()
for p in image_clf.parameters():
    p.requires_grad = False

# Feature extractor: same backbone, Identity head -> 2048-d
image_encoder = build_resnet50(num_classes=2)
image_encoder.load_state_dict(img_ckpt["model_state_dict"])
IMAGE_DIM = image_encoder.fc.in_features
image_encoder.fc = nn.Identity()
image_encoder.to(DEVICE).eval()
for p in image_encoder.parameters():
    p.requires_grad = False

eval_tf = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

print(f"TEXT_DIM={TEXT_DIM}  IMAGE_DIM={IMAGE_DIM}")


## Text cleaning (match Module 01)

Row-local cleaner only — same rules as the text branch.


In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")


def clean_text(text) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text


## Extract and cache embeddings

Save `{split}_text.npy`, `{split}_image.npy`, `{split}_labels.npy`, plus unimodal probability matrices for the agreement explainer. Re-run with `FORCE_REBUILD_EMB = True` after changing the subsample.


In [ ]:
FORCE_REBUILD_EMB = False
EMB_BATCH = 32 if DEVICE == "cuda" else 8


@torch.no_grad()
def encode_text_batch(texts):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = text_model.distilbert(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
    )
    # DistilBERT pooled token (position 0), then pre_classifier path used by the clf head
    hidden = out.last_hidden_state[:, 0]
    pooled = text_model.pre_classifier(hidden)
    pooled = F.relu(pooled)
    logits = text_model.classifier(pooled)
    probs = F.softmax(logits, dim=-1)
    return pooled.cpu().numpy(), probs.cpu().numpy()


@torch.no_grad()
def encode_image_batch(paths):
    tensors = []
    for p in paths:
        with Image.open(p) as im:
            tensors.append(eval_tf(im.convert("RGB")))
    batch = torch.stack(tensors, dim=0).to(DEVICE)
    feats = image_encoder(batch)
    logits = image_clf(batch)
    probs = F.softmax(logits, dim=-1)
    return feats.cpu().numpy(), probs.cpu().numpy()


def build_or_load_embeddings(df: pd.DataFrame, split: str):
    t_path = os.path.join(EMB_CACHE, f"{split}_text.npy")
    i_path = os.path.join(EMB_CACHE, f"{split}_image.npy")
    y_path = os.path.join(EMB_CACHE, f"{split}_labels.npy")
    tp_path = os.path.join(EMB_CACHE, f"{split}_text_probs.npy")
    ip_path = os.path.join(EMB_CACHE, f"{split}_image_probs.npy")
    id_path = os.path.join(EMB_CACHE, f"{split}_ids.npy")

    needed = [t_path, i_path, y_path, tp_path, ip_path, id_path]
    if (not FORCE_REBUILD_EMB) and all(os.path.exists(p) for p in needed):
        print(f"Loaded cached embeddings: {split}")
        return {
            "text": np.load(t_path),
            "image": np.load(i_path),
            "y": np.load(y_path),
            "text_probs": np.load(tp_path),
            "image_probs": np.load(ip_path),
            "ids": np.load(id_path, allow_pickle=True),
        }

    texts = df[TEXT_COL].map(clean_text).tolist()
    paths = df["image_path"].tolist()
    labels = df[LABEL_COL].astype(int).to_numpy()
    ids = df[ID_COL].astype(str).to_numpy()

    text_embs, text_probs, img_embs, img_probs = [], [], [], []
    for start in tqdm(range(0, len(df), EMB_BATCH), desc=f"embed:{split}"):
        end = start + EMB_BATCH
        te, tp = encode_text_batch(texts[start:end])
        ie, ip = encode_image_batch(paths[start:end])
        text_embs.append(te)
        text_probs.append(tp)
        img_embs.append(ie)
        img_probs.append(ip)

    pack = {
        "text": np.concatenate(text_embs, axis=0).astype(np.float32),
        "image": np.concatenate(img_embs, axis=0).astype(np.float32),
        "y": labels.astype(np.int64),
        "text_probs": np.concatenate(text_probs, axis=0).astype(np.float32),
        "image_probs": np.concatenate(img_probs, axis=0).astype(np.float32),
        "ids": ids,
    }
    np.save(t_path, pack["text"])
    np.save(i_path, pack["image"])
    np.save(y_path, pack["y"])
    np.save(tp_path, pack["text_probs"])
    np.save(ip_path, pack["image_probs"])
    np.save(id_path, pack["ids"])
    print(f"Wrote embeddings → {split}  text{pack['text'].shape}  image{pack['image'].shape}")
    return pack


train_emb = build_or_load_embeddings(train_df, "train")
val_emb = build_or_load_embeddings(val_df, "val")
test_emb = build_or_load_embeddings(test_df, "test")


## Fusion models

**LateFusionMLP** — concat(text, image) → MLP.  
**CrossAttentionFusion** — project both modalities to a shared width, bidirectional attention on length-1 sequences, then classify.


In [ ]:
class EmbeddingPairDataset(Dataset):
    def __init__(self, pack: dict):
        self.text = torch.from_numpy(pack["text"])
        self.image = torch.from_numpy(pack["image"])
        self.y = torch.from_numpy(pack["y"]).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.text[idx], self.image[idx], self.y[idx]


class LateFusionMLP(nn.Module):
    def __init__(self, text_dim: int, image_dim: int, hidden: int = 512, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 2),
        )

    def forward(self, text_emb, image_emb):
        x = torch.cat([text_emb, image_emb], dim=-1)
        return self.net(x)


class CrossAttentionFusion(nn.Module):
    def __init__(self, text_dim: int, image_dim: int, d_model: int = 256, nhead: int = 4, dropout: float = 0.3):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, d_model)
        self.image_proj = nn.Linear(image_dim, d_model)
        self.text_to_image = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.image_to_text = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 2),
        )

    def forward(self, text_emb, image_emb, return_attn: bool = False):
        t = self.text_proj(text_emb).unsqueeze(1)   # (B, 1, D)
        i = self.image_proj(image_emb).unsqueeze(1)
        t2i, attn_t2i = self.text_to_image(t, i, i, need_weights=True, average_attn_weights=True)
        i2t, attn_i2t = self.image_to_text(i, t, t, need_weights=True, average_attn_weights=True)
        fused = torch.cat([t.squeeze(1), i.squeeze(1), t2i.squeeze(1), i2t.squeeze(1)], dim=-1)
        logits = self.classifier(fused)
        if return_attn:
            return logits, attn_t2i, attn_i2t
        return logits


BATCH_SIZE = 64
train_loader = DataLoader(EmbeddingPairDataset(train_emb), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmbeddingPairDataset(val_emb), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(EmbeddingPairDataset(test_emb), batch_size=BATCH_SIZE, shuffle=False)

print(f"batches → train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}")


## Train fusion heads

Encoders stay frozen. Class-weighted cross-entropy; keep the best validation macro-F1 checkpoint for each fusion variant.


In [ ]:
EPOCHS = 12
LR = 1e-3
WEIGHT_DECAY = 1e-4

counts = pd.Series(train_emb["y"]).value_counts().sort_index()
n_total = float(counts.sum())
class_weights = torch.tensor(
    [n_total / (2.0 * float(counts.get(0, 1))), n_total / (2.0 * float(counts.get(1, 1)))],
    dtype=torch.float32,
    device=DEVICE,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("class_weights:", class_weights.tolist())


@torch.no_grad()
def predict_fusion(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for text_e, image_e, y in loader:
        text_e = text_e.to(DEVICE)
        image_e = image_e.to(DEVICE)
        logits = model(text_e, image_e)
        p = F.softmax(logits, dim=-1)
        ys.append(y.numpy())
        preds.append(logits.argmax(dim=1).cpu().numpy())
        probs.append(p.cpu().numpy())
    return np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)


def train_fusion(model: nn.Module, name: str):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_state, best_f1 = None, -1.0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, n = 0.0, 0
        for text_e, image_e, y in train_loader:
            text_e = text_e.to(DEVICE)
            image_e = image_e.to(DEVICE)
            y = y.to(DEVICE)
            logits = model(text_e, image_e)
            loss = criterion(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total_loss += float(loss.item()) * y.size(0)
            n += y.size(0)

        y_true, y_pred, _ = predict_fusion(model, val_loader)
        val_acc = accuracy_score(y_true, y_pred)
        val_f1 = f1_score(y_true, y_pred, average="macro")
        history.append({"epoch": epoch, "train_loss": total_loss / max(n, 1), "val_acc": val_acc, "val_f1": val_f1})
        print(f"[{name}] epoch {epoch}/{EPOCHS}  loss={total_loss/max(n,1):.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ↑ best val macro-F1 = {best_f1:.4f}")

    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, history, best_f1


late_model, late_hist, late_best = train_fusion(
    LateFusionMLP(TEXT_DIM, IMAGE_DIM), "late_fusion"
)
xattn_model, xattn_hist, xattn_best = train_fusion(
    CrossAttentionFusion(TEXT_DIM, IMAGE_DIM), "cross_attn"
)


## Evaluate vs unimodal baselines

Multimodal should beat DistilBERT-only and ResNet50-only on this paired test subset. Also report unimodal votes from the frozen heads for the same IDs.


In [ ]:
def evaluate_classifier(name: str, y_true, y_pred) -> dict:
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    print(f"\n===== {name} =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 macro: {f1_macro:.4f} | F1 weighted: {f1_weighted:.4f}")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(name)
    plt.tight_layout()
    plt.show()
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}


# Unimodal on the same paired test subset
uni_text_pred = test_emb["text_probs"].argmax(axis=1)
uni_image_pred = test_emb["image_probs"].argmax(axis=1)
metrics_text = evaluate_classifier("DistilBERT (paired test)", test_emb["y"], uni_text_pred)
metrics_image = evaluate_classifier("ResNet50 (paired test)", test_emb["y"], uni_image_pred)

y_late, pred_late, prob_late = predict_fusion(late_model, test_loader)
y_xattn, pred_xattn, prob_xattn = predict_fusion(xattn_model, test_loader)
# val metrics
yv_late, pv_late, _ = predict_fusion(late_model, val_loader)
yv_xattn, pv_xattn, _ = predict_fusion(xattn_model, val_loader)

metrics_late_test = evaluate_classifier("Late fusion — TEST", y_late, pred_late)
metrics_xattn_test = evaluate_classifier("Cross-attn fusion — TEST", y_xattn, pred_xattn)
metrics_late_val = evaluate_classifier("Late fusion — VAL", yv_late, pv_late)
metrics_xattn_val = evaluate_classifier("Cross-attn fusion — VAL", yv_xattn, pv_xattn)

print("\n--- Beat unimodal? (test macro-F1) ---")
print(f"text={metrics_text['f1_macro']:.4f}  image={metrics_image['f1_macro']:.4f}  "
      f"late={metrics_late_test['f1_macro']:.4f}  xattn={metrics_xattn_test['f1_macro']:.4f}")


## Agreement score + natural-language explainer

Cosine similarity between L2-normalized text and image embeddings. Combine with unimodal confidences to name which modality drove the fused decision when they disagree.


In [ ]:
def agreement_scores(text_emb: np.ndarray, image_emb: np.ndarray) -> np.ndarray:
    # Pairwise diagonal cosine between aligned rows
    t = text_emb / (np.linalg.norm(text_emb, axis=1, keepdims=True) + 1e-8)
    i = image_emb / (np.linalg.norm(image_emb, axis=1, keepdims=True) + 1e-8)
    return np.sum(t * i, axis=1)


def explain_prediction(
    fusion_pred: int,
    fusion_prob: np.ndarray,
    text_prob: np.ndarray,
    image_prob: np.ndarray,
    agreement: float,
    low_agree: float = 0.15,
) -> str:
    """Rule-based one-liner. Agreement alone is never the verdict."""
    label = LABEL_NAMES[fusion_pred]
    conf = float(fusion_prob[fusion_pred])
    text_pred = int(np.argmax(text_prob))
    image_pred = int(np.argmax(image_prob))
    text_conf = float(text_prob[text_pred])
    image_conf = float(image_prob[image_pred])

    modalities_agree = text_pred == image_pred
    low = agreement < low_agree

    if modalities_agree and text_pred == fusion_pred:
        return (
            f"Predicted {label} (conf={conf:.2f}). Text and image agree "
            f"(agreement={agreement:.2f}); both support {LABEL_NAMES[text_pred]}."
        )
    if low and image_conf >= text_conf:
        return (
            f"Predicted {label} (conf={conf:.2f}). Low text–image agreement ({agreement:.2f}); "
            f"decision driven mainly by the image (image→{LABEL_NAMES[image_pred]} @ {image_conf:.2f})."
        )
    if low and text_conf > image_conf:
        return (
            f"Predicted {label} (conf={conf:.2f}). Low text–image agreement ({agreement:.2f}); "
            f"decision driven mainly by the wording/style "
            f"(text→{LABEL_NAMES[text_pred]} @ {text_conf:.2f})."
        )
    if text_pred != image_pred:
        driver = "image" if image_conf >= text_conf else "text"
        return (
            f"Predicted {label} (conf={conf:.2f}). Modalities disagree "
            f"(text→{LABEL_NAMES[text_pred]}, image→{LABEL_NAMES[image_pred]}, "
            f"agreement={agreement:.2f}); fused decision leans on the {driver} branch."
        )
    return (
        f"Predicted {label} (conf={conf:.2f}) with agreement={agreement:.2f}. "
        f"Text={LABEL_NAMES[text_pred]} ({text_conf:.2f}), "
        f"image={LABEL_NAMES[image_pred]} ({image_conf:.2f})."
    )


test_agree = agreement_scores(test_emb["text"], test_emb["image"])
print(f"Test agreement: mean={test_agree.mean():.3f}  std={test_agree.std():.3f}")

# Prefer a disagreeing / low-agreement example for a clear demo
disagree_idx = [
    i for i in range(len(test_emb["y"]))
    if test_emb["text_probs"][i].argmax() != test_emb["image_probs"][i].argmax()
]
demo_indices = disagree_idx[:3] if disagree_idx else list(range(min(3, len(test_emb["y"]))))

print("\nSample explanations (late fusion):")
for i in demo_indices:
    msg = explain_prediction(
        fusion_pred=int(pred_late[i]),
        fusion_prob=prob_late[i],
        text_prob=test_emb["text_probs"][i],
        image_prob=test_emb["image_probs"][i],
        agreement=float(test_agree[i]),
    )
    print(f"- id={test_emb['ids'][i]}  true={LABEL_NAMES[int(test_emb['y'][i])]}")
    print(f"  {msg}")


## Save fusion artifacts


In [ ]:
SAVE_DIR = os.path.join(DATA_DIR, "checkpoints", "module03_fusion")
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(
    {
        "late_fusion_state_dict": late_model.state_dict(),
        "cross_attn_state_dict": xattn_model.state_dict(),
        "text_dim": TEXT_DIM,
        "image_dim": IMAGE_DIM,
        "label_names": LABEL_NAMES,
        "late_best_val_f1": late_best,
        "xattn_best_val_f1": xattn_best,
        "late_history": late_hist,
        "xattn_history": xattn_hist,
    },
    os.path.join(SAVE_DIR, "fusion.pt"),
)

meta = {
    "text_ckpt": TEXT_CKPT,
    "image_ckpt": IMAGE_CKPT,
    "subsample_train": SUBSAMPLE_TRAIN,
    "subsample_val": SUBSAMPLE_VAL,
    "subsample_test": SUBSAMPLE_TEST,
    "epochs": EPOCHS,
    "lr": LR,
    "metrics_test": {
        "distilbert_paired": metrics_text,
        "resnet50_paired": metrics_image,
        "late_fusion": metrics_late_test,
        "cross_attn_fusion": metrics_xattn_test,
    },
}
with open(os.path.join(SAVE_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

summary = pd.DataFrame(
    [
        {"model": "DistilBERT (paired)", "split": "test", **metrics_text},
        {"model": "ResNet50 (paired)", "split": "test", **metrics_image},
        {"model": "Late fusion", "split": "test", **metrics_late_test},
        {"model": "Cross-attn fusion", "split": "test", **metrics_xattn_test},
        {"model": "Late fusion", "split": "val", **metrics_late_val},
        {"model": "Cross-attn fusion", "split": "val", **metrics_xattn_val},
    ]
)
display(summary)

# Reference Module 01/02 full-split metrics if present
for path, title in [(M1_METRICS, "Module 01"), (M2_METRICS, "Module 02")]:
    if os.path.exists(path):
        print(f"\n{title} metrics file:")
        display(pd.read_csv(path))

summary_path = os.path.join(DATA_DIR, "checkpoints", "module03_metrics.csv")
summary.to_csv(summary_path, index=False)
print(f"Wrote {summary_path}")
print(f"Saved fusion weights → {os.path.join(SAVE_DIR, 'fusion.pt')}")


## Artifacts

| Output | Location |
|--------|----------|
| Fusion weights (late + cross-attn) | `{DATA_DIR}/checkpoints/module03_fusion/fusion.pt` |
| Run config | `{DATA_DIR}/checkpoints/module03_fusion/run_config.json` |
| Cached embeddings | `{DATA_DIR}/checkpoints/module03_embeddings/` |
| Metrics table | `{DATA_DIR}/checkpoints/module03_metrics.csv` |

Commit only `module03_metrics.csv` to git (same pattern as Modules 01–02). Keep `fusion.pt` and embeddings on Drive.

**Next (Module 04):** faithfulness deletion tests on SHAP / Grad-CAM regions, missing-modality ablation, and the full comparison table.

If multimodal F1 does **not** beat both unimodal paired scores, debug fusion (overfitting, bad pairs, too-small subsample) before adding features.
